In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 228 (delta 101), reused 179 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (228/228), 797.23 KiB | 4.98 MiB/s, done.
Resolving deltas: 100% (101/101), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 246.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 349.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 251.9 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 50.5 MB/s eta 0:00:00


In [4]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class SeeMoreBlocksRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()

    def reset(self) -> None:
        self._blocks_seen = set()

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    return 1.0
        return 0.0

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

class CNNNeuralPolicy(nn.Module):
    def __init__(self, num_actions):
        super().__init__()

        # Input shape: (16, 14, 16) -> (Channels, Height, Width)
        self.conv_layers = nn.Sequential(
            # First layer: Looking for basic patterns
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            # Second layer: Combining features
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            # Pooling to reduce dimensionality slightly
            nn.MaxPool2d(kernel_size=2)
        )

        # After MaxPool(2x2), 14x16 becomes 7x8
        # Flattened size = 64 channels * 7 height * 8 width = 3584
        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions)
        )

    def forward(self, x):
        # Convert numpy observation to tensor and normalize if needed
        # Your ApplyEncodingWrapper outputs uint8 (0-255)
        x = torch.from_numpy(x).float().to(device) / 255.0

        if x.ndimension() == 3:
            x = x.unsqueeze(0) # Add batch dimension: (1, 16, 14, 16)

        x = self.conv_layers(x)
        x = self.flatten(x)
        logits = self.fc(x)

        # For GA, we just want the best action index
        return torch.argmax(logits, dim=1).item()

    def mutate(self, rate=0.1, sigma=0.1):
        """Adds Gaussian noise to weights/biases for evolution."""
        with torch.no_grad():
            for param in self.parameters():
                if torch.rand(1).item() < rate:
                    noise = torch.randn_like(param) * sigma
                    param.add_(noise)

In [8]:
RUN_NAME = "tile_finder"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 3

In [9]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [10]:
def evaluate_individual(individual, env):
    obs, info = env.reset()
    total_reward = 0
    done = False
    with torch.no_grad():
        while not done:
            action = individual.forward(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
    return total_reward

In [11]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = SeeMoreBlocksRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
encoder = TileEncoder()

2026-05-04 20:55:19 [INFO] Session log for run tile_finder with level [INFO] initialized at: tile_finder_20260504_205519.log
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
try:
    distinct_observations = []
    last_distinct_obs = None
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(SuperMarioCombo.get_combo_id(SuperMarioCombo.RIGHT))
        if last_distinct_obs is None or not np.array_equal(obs, last_distinct_obs):
            last_distinct_obs = obs
            distinct_observations.append(obs)
        done = terminated or truncated
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [13]:
encoder.train(distinct_observations, print_progress=True)

Epoch 5/5: 100%|██████████| 5/5 [00:00<00:00, 62.16it/s, loss=5.52]


In [14]:
class ApplyEncodingWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, encoder: TileEncoder, device: Optional[str] = None):
        super().__init__(env)
        self._encoder = encoder
        self.device = device
        if self.device is not None:
            self._encoder.to(device)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(16, 14, 16),
            dtype=np.uint8
        )

    def observation(self, observation: np.ndarray) -> np.ndarray:
        features = self._encoder.embed(observation)
        features = features.permute(2, 0, 1).cpu().numpy()
        return (features * 255).astype(np.uint8)

In [15]:
# Hyperparameters
POPULATION_SIZE = 12
GENERATIONS = 50
MUTATION_RATE = 0.1
MUTATION_SIGMA = 0.2

In [16]:
# Initialize Environment and Population
encoded_env = ApplyEncodingWrapper(env, encoder)
num_actions = len(SuperMarioCombo)
population = [CNNNeuralPolicy(num_actions).to(device) for _ in range(POPULATION_SIZE)]

In [ ]:
logger.info(f"Starting Genetic Algorithm evolution for {GENERATIONS} generations...")

for gen in range(GENERATIONS):
    # 1. Evaluate Fitness
    fitness_scores = []
    for i, individual in enumerate(population):
        fitness = evaluate_individual(individual, encoded_env)
        fitness_scores.append((fitness, i))

    # Sort by fitness (descending)
    fitness_scores.sort(key=lambda x: x[0], reverse=True)
    best_fitness = fitness_scores[0][0]

    logger.info(f"Gen {gen} | Best Fitness: {best_fitness} | Avg: {np.mean([f[0] for f in fitness_scores])}")

    # 2. Selection (Keep the top 25% as parents)
    top_performers_indices = [idx for score, idx in fitness_scores[:POPULATION_SIZE // 4]]
    parents = [population[idx] for idx in top_performers_indices]

    # 3. Reproduction & Mutation
    new_population = []
    # Keep the absolute best (Elitism)
    new_population.append(copy.deepcopy(parents[0]))

    while len(new_population) < POPULATION_SIZE:
        # Pick a random parent and clone it
        parent = copy.deepcopy(np.random.choice(parents))
        parent.mutate(rate=MUTATION_RATE, sigma=MUTATION_SIGMA)
        new_population.append(parent)

    population = new_population

2026-05-04 20:55:43 [INFO] Starting Genetic Algorithm evolution for 50 generations...
2026-05-04 21:32:07 [INFO] Gen 0 | Best Fitness: 7.0 | Avg: 7.0


In [ ]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = SeeMoreBlocksRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
encoder = TileEncoder()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    encoded_env = ApplyEncodingWrapper(env, encoder)
    video_env = RecordVideo(encoded_env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()
#     del env
#     import gc
#     gc.collect()

In [ ]:
try:
    distinct_observations = []
    last_distinct_obs = None
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(SuperMarioCombo.get_combo_id(SuperMarioCombo.RIGHT))
        if last_distinct_obs is None or not np.array_equal(obs, last_distinct_obs):
            last_distinct_obs = obs
            distinct_observations.append(obs)
        done = terminated or truncated
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
encoder.train(distinct_observations, print_progress=True)

In [ ]:
class ApplyEncodingWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, encoder: TileEncoder, device: Optional[str] = None):
        super().__init__(env)
        self._encoder = encoder
        self.device = device
        if self.device is not None:
            self._encoder.to(device)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(16, 14, 16),
            dtype=np.uint8
        )

    def observation(self, observation: np.ndarray) -> np.ndarray:
        features = self._encoder.embed(observation)
        features = features.permute(2, 0, 1).cpu().numpy()
        return (features * 255).astype(np.uint8)

In [ ]:
try:
    encoded_env = ApplyEncodingWrapper(env, encoder)
    ppo_env = DummyVecEnv([lambda: encoded_env])
    model = PPO("MlpPolicy", ppo_env, verbose=1, learning_rate=0.0003, n_steps=10000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    encoded_env = ApplyEncodingWrapper(env, encoder)
    video_env = RecordVideo(encoded_env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()
#     del env
#     import gc
#     gc.collect()